## User story
Build a StoryMap automatically from a repeatable weekly report (title, sections, credits, images, and a link), where the content lives in consistent locations in your org.

## What this notebook does
1. Signs in to ArcGIS Online (or your portal) using `.env` credentials.
2. Reads a CSV that describes the story text blocks (title, headings, paragraphs, credits).
3. Creates a new StoryMap and appends those blocks in order.
4. Adds a cover image, a hero image, and an embedded link card.
5. Publishes the story.

## Before you run it
- Create a `.env` file (or set env vars) with `USERNAME_TARGET` and `PASSWORD_TARGET`.
- Put these files in the `project_folder` path (configured below):
  - `copy_edit.csv`
  - `Cover.png`
  - `Hero.png`

In [ ]:
import arcgis
from arcgis.gis import GIS
from arcgis.apps.storymap import StoryMap
from arcgis.apps.storymap.story_content import Text, Image, TextStyles, Embed
from dotenv import load_dotenv
from pathlib import Path
import csv
import os
from typing import NamedTuple

In [ ]:
EXPECTED_ARCGIS_VERSION = "2.4.3"
print("ArcGIS API for Python version:", arcgis.__version__)
if arcgis.__version__ != EXPECTED_ARCGIS_VERSION:
    print(
        "Note: this notebook was authored against",
        EXPECTED_ARCGIS_VERSION,
        "(you have",
        arcgis.__version__ + ")",
    )

In [ ]:
# Load environment variables from .env (or your OS environment)
load_dotenv()

USERNAME_TARGET = os.getenv("USERNAME_TARGET")
PASSWORD_TARGET = os.getenv("PASSWORD_TARGET")

if not USERNAME_TARGET or not PASSWORD_TARGET:
    raise ValueError(
        "Missing credentials. Set USERNAME_TARGET and PASSWORD_TARGET in a .env file (or env vars)."
    )

In [ ]:
#login to arcgis
gis_con = GIS("https://www.arcgis.com", USERNAME_TARGET, PASSWORD_TARGET)

In [ ]:
class Section(NamedTuple):
    content_type: str
    content: str
    approved: str

In [ ]:
# Folder containing the CSV and images for this example
project_folder = Path("projects") / "County Park"

csv_path = project_folder / "copy_edit.csv"
cover_image_path = project_folder / "Cover.png"
hero_image_path = project_folder / "Hero.png"

## Load story text from CSV
This notebook expects a CSV with **three columns**:
- `content_type`: The kind of block to add (e.g., `Title`, `Credits`, or a heading like `Overview`).
- `content`: The text to place in that block.
- `approved`: A flag you can use in your process (this notebook loads it, but doesn’t filter on it by default).

### Example CSV
```csv
content_type,content,approved
Title,County Park weekly report,TRUE
Overview,This week we completed trail maintenance.,TRUE
Credits,"Alex:Project Manager; Sam:GIS Analyst",TRUE
```

Tip: Using a real CSV parser (instead of splitting on commas) means your text can safely include commas when quoted.

In [ ]:
# Read CSV rows into structured sections
if not csv_path.exists():
    raise FileNotFoundError(f"CSV not found: {csv_path}")

text_sections: list[Section] = []
with csv_path.open("r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    required_fields = {"content_type", "content", "approved"}
    if not reader.fieldnames or not required_fields.issubset(set(reader.fieldnames)):
        raise ValueError(
            f"CSV must include columns {sorted(required_fields)}. Found: {reader.fieldnames}"
        )
    for row in reader:
        text_sections.append(
            Section(
                content_type=(row.get("content_type") or "").strip(),
                content=(row.get("content") or "").strip(),
                approved=(row.get("approved") or "").strip(),
            )
        )

# Preview parsed rows
for section in text_sections:
    print(section)

## Helper functions
These helpers translate CSV rows into StoryMap blocks.
- `add_title`: Updates the story cover title (assumes the cover is the first block).
- `add_heading_and_text`: Adds a heading followed by paragraph text.
- `add_credit_section`: Parses a `name:role; name:role` string and adds credits rows.

In [ ]:
def add_title(story: StoryMap, content: str) -> None:
    cover = story.contents[0]
    cover.title = content


def add_heading_and_text(story: StoryMap, content: str, heading_text: str) -> None:
    heading_block = Text(text=heading_text, style=TextStyles.HEADING)
    paragraph_block = Text(text=content, style=TextStyles.PARAGRAPH)
    story.add(content=heading_block)
    story.add(content=paragraph_block)


def add_credit_section(story: StoryMap, content: str) -> None:
    # Expected format: "Name:Role; Name:Role"
    for team_member in filter(None, (c.strip() for c in content.split(";"))):
        if ":" not in team_member:
            continue
        name, position = (p.strip() for p in team_member.split(":", 1))
        story.credits(
            content=name,
            attribution=position,
            heading="Team Members",
            description="Meet the team that is proud to work on this project.",
        )

In [ ]:
#initialize story
new_story = StoryMap(gis=gis_con)

In [ ]:
new_story.contents


### Reading and Using Your Story Content

#### Option 1

- List of all blocks in your story
- At any time, you can reference a content block by its index
- Then modify its attributes

<br>

OR

<br>

#### Option 2

- Create a new block instance of the appropriate type
- Set its attributes
- Add it to your story at your chosen index

### Adding the text content to the storymap

In [ ]:
# Add text blocks (in CSV order)
for section in text_sections:
    if section.content_type == "Title":
        add_title(story=new_story, content=section.content)
    elif section.content_type == "Credits":
        add_credit_section(story=new_story, content=section.content)
    else:
        add_heading_and_text(
            story=new_story,
            content=section.content,
            heading_text=section.content_type,
        )

In [ ]:
new_story.contents

### Add the Cover Image

1. Creates an Image object with the provided path.
2. Sets the display width.
3. Gets the cover object from the storymap content.
4. Sets the media attribute of the cover object to the created Image object.

In [ ]:
# Add cover image
if not cover_image_path.exists():
    raise FileNotFoundError(f"Cover image not found: {cover_image_path}")

cover_image = Image(str(cover_image_path))
cover_image.display = "wide"
cover = new_story.contents[0]
cover.media = cover_image
cover.media

### Add the Hero Image

1. Creates an Image object with the provided path.
2. Sets the display width for the Image object.
3. Adds the Image object to the storymap at the specified index.

In [ ]:
# Add hero image
# Note: position=4 is specific to this example's block order.
if not hero_image_path.exists():
    raise FileNotFoundError(f"Hero image not found: {hero_image_path}")

hero_image = Image(str(hero_image_path))
hero_image.display = "standard"
new_story.add(content=hero_image, position=4)

In [ ]:
new_story.contents

In [ ]:
## Add our site url as a card to the end
embed = Embed(path="https://www.example.com")
embed.display ='card'
embed.caption = "Project Website"
new_story.add(content=embed, position=-1)

In [ ]:
new_story.contents

In [ ]:
new_story.save(publish=True)